### Проверка `lakehouse.stage.case13_match_players` (уникальность, покрытие и пропуски)

Проверяем отсутствие дубликатов по первичному ключу `(match_id, player_slot)` и корректность разметки флагами `in_pg_matches` и `match_has_tde_events`.

**Выводы:**
* Всего строк: 4 834 482 | Дубликатов `(match_id, player_slot)`: 0.
* Согласованное ядро (`in_pg_matches = true`, без пропусков в `duration` и `hero_name`):
  * `Standard` с событиями в Mongo: 217 331 матч (2 173 310 строк).
  * `Turbo` с событиями в Mongo: 103 382 матча (1 033 820 строк).
  * Матчи без событий урона по строениям (`match_has_tde_events = false`): 850 в `Standard` и 66 в `Turbo`.
* Изолированные «сиротские» записи (`in_pg_matches = false`): 161 828 матчей (1 618 192 строки) — оставлены в `STAGE` для прозрачности DQ и отсекаются при построении `MART`.

In [7]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window

with SparkSessionContext("lakehouse.stage_case13") as spark:
        
    st_mp = spark.table("lakehouse.stage.case13_match_players")
    
    print("=== 1. Размер и уникальность (match_id, player_slot) ===")
    dup_mp = st_mp.groupBy("match_id", "player_slot").count().filter(F.col("count") > 1).count()
    print(f"Total rows: {st_mp.count():,} | Duplicate (match_id, player_slot): {dup_mp}")
    
    print("=== 2. Покрытие и пропуски ===")
    st_mp.groupBy("in_pg_matches", "match_has_tde_events", "game_mode_group").agg(
        F.count("*").alias("rows"),
        F.countDistinct("match_id").alias("matches"),
        F.sum(F.when(F.col("duration").isNull(), 1).otherwise(0)).alias("null_duration"),
        F.sum(F.when(F.col("hero_name").isNull(), 1).otherwise(0)).alias("null_hero_name")
    ).orderBy(F.desc("matches")).show(20, truncate=False)

=== 1. Размер и уникальность (match_id, player_slot) ===


Total rows: 4,834,482 | Duplicate (match_id, player_slot): 0
=== 2. Покрытие и пропуски ===


+-------------+--------------------+---------------+-------+-------+-------------+--------------+
|in_pg_matches|match_has_tde_events|game_mode_group|rows   |matches|null_duration|null_hero_name|
+-------------+--------------------+---------------+-------+-------+-------------+--------------+
|true         |true                |Standard       |2173310|217331 |0            |0             |
|false        |false               |NULL           |1609142|160923 |1609142      |10417         |
|true         |true                |Turbo          |1033820|103382 |0            |0             |
|false        |true                |NULL           |9050   |905    |9050         |0             |
|true         |false               |Standard       |8500   |850    |0            |0             |
|true         |false               |Turbo          |660    |66     |0            |0             |
+-------------+--------------------+---------------+-------+-------+-------------+--------------+



### Проверка `lakehouse.stage.case13_tower_damage` (типы строений, механики и фазы)

Анализируем распределение 246 662 714 обогащённых событий урона (10 269 548 753 суммарного урона) по классифицированным категориям.

**Выводы:**
* **По типу строения (`building_category`):** `TOWER` — 161 622 570 событий (5,90 млрд урона, 57,41%), `BARRACKS` — 25,78%, `FORT` — 9,18%, `FILLER` — 7,63%. Неклассифицированных построек (`OTHER`) нет (0 строк).
* **По механике урона внутри башен (`TOWER`):**
  * `auto_attack`: 115 377 058 событий (4,99 млрд урона, 84,68%).
  * `summon_without_ability` (`fromnpc > 0, byability = 0`): 22 429 106 событий (392,30 млн урона, 6,65%).
  * `ability_direct`: 14 077 247 событий (348,16 млн урона, 5,91%).
  * `ability_summon`: 8 601 321 событие (92,32 млн урона, 1,57%).
  * `item`: 1 137 838 событий (70,70 млн урона, 1,20%).
* **По фазам матча внутри башен (`TOWER`):** `0-10` - 337,18 млн урона, `10-20` - 1,43 млрд, `20-30` - 1,78 млрд, `30+` - 2,35 млрд.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window

with SparkSessionContext("lakehouse.stage_case13") as spark:
    
    st_tde = spark.table("lakehouse.stage.case13_tower_damage")
    raw_tde = spark.table("lakehouse.raw.mongo__tower_damage_events")
    
    print("=== 3. Распределение по building_category, damage_source_type, phase ===")
    w_all = Window.partitionBy()
    
    st_tde.groupBy("building_category").agg(
        F.count("*").alias("events"),
        F.sum(F.col("damage").cast("long")).alias("dmg")
    ).withColumn("dmg_pct", F.round(F.col("dmg") * 100.0 / F.sum("dmg").over(w_all), 2)) \
     .orderBy(F.desc("dmg")).show(truncate=False)
    
    st_tde.filter(F.col("building_category") == "TOWER").groupBy("damage_source_type").agg(
        F.count("*").alias("events"),
        F.sum(F.col("damage").cast("long")).alias("dmg")
    ).withColumn("dmg_pct", F.round(F.col("dmg") * 100.0 / F.sum("dmg").over(w_all), 2)) \
     .orderBy(F.desc("dmg")).show(truncate=False)
    
    st_tde.filter(F.col("building_category") == "TOWER").groupBy("phase").agg(
        F.count("*").alias("events"),
        F.sum(F.col("damage").cast("long")).alias("dmg")
    ).orderBy("phase").show(truncate=False)

=== 3. Распределение по building_category, damage_source_type, phase ===


26/11/14 08:47:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:47:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:47:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:48:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:48:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:48:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 0

+-----------------+---------+----------+-------+
|building_category|events   |dmg       |dmg_pct|
+-----------------+---------+----------+-------+
|TOWER            |161622570|5895541007|57.41  |
|BARRACKS         |47513112 |2647668634|25.78  |
|FORT             |20198348 |942271735 |9.18   |
|FILLER           |17328684 |784067377 |7.63   |
+-----------------+---------+----------+-------+



26/11/14 08:48:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:48:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:48:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:48:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:48:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:48:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 0

+----------------------+---------+----------+-------+
|damage_source_type    |events   |dmg       |dmg_pct|
+----------------------+---------+----------+-------+
|auto_attack           |115377058|4992066401|84.68  |
|summon_without_ability|22429106 |392301878 |6.65   |
|ability_direct        |14077247 |348157138 |5.91   |
|ability_summon        |8601321  |92316210  |1.57   |
|item                  |1137838  |70699380  |1.2    |
+----------------------+---------+----------+-------+



+-----+--------+----------+
|phase|events  |dmg       |
+-----+--------+----------+
|0-10 |14943493|337184053 |
|10-20|46261724|1429967962|
|20-30|47252927|1776991546|
|30+  |53164426|2351397446|
+-----+--------+----------+



### Глобальная сверка урона между `case13_tower_damage` и `case13_match_players`

Сравниваем сумму событийного урона по официальным постройкам (`TOWER + BARRACKS + FORT`) с эталонным агрегатом `pg__match_players.tower_damage` на согласованном срезе матчей.

**Выводы:**
* Событийный урон (`case13_tower_damage`): 9 430 978 984 (320 386 матчей).
* Эталон Postgres (`case13_match_players`): 9 444 923 914 (320 713 матчей).
* Расхождение составило **-0,1476%** (разница в 327 матчей вызвана асинхронным пополнением `RAW` между запусками двух потоков). Это подтверждает отсутствие дублирования строк при `JOIN` в `STAGE`.

In [1]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window

with SparkSessionContext("lakehouse.stage_case13") as spark:
    
    st_mp = spark.table("lakehouse.stage.case13_match_players")
    st_tde = spark.table("lakehouse.stage.case13_tower_damage")
    
    tde_agg = st_tde.filter(
        (F.col("is_matched_match") == True) &
        (F.col("building_category").isin("TOWER", "BARRACKS", "FORT"))
    ).agg(
        F.sum(F.col("damage").cast("long")).alias("stage_tde_official_dmg"),
        F.countDistinct("match_id").alias("tde_matches")
    )
    
    mp_agg = st_mp.filter(
        (F.col("in_pg_matches") == True) &
        (F.col("match_has_tde_events") == True)
    ).agg(
        F.sum(F.col("tower_damage").cast("long")).alias("stage_mp_tower_dmg"),
        F.countDistinct("match_id").alias("mp_matches")
    )
    
    tde_agg.crossJoin(mp_agg).withColumn(
        "diff_pct",
        F.round((F.col("stage_tde_official_dmg") - F.col("stage_mp_tower_dmg")) * 100.0 / F.col("stage_mp_tower_dmg"), 4)
    ).show(truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 08:46:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+----------------------+-----------+------------------+----------+--------+
|stage_tde_official_dmg|tde_matches|stage_mp_tower_dmg|mp_matches|diff_pct|
+----------------------+-----------+------------------+----------+--------+
|9430978984            |320386     |9444923914        |320713    |-0.1476 |
+----------------------+-----------+------------------+----------+--------+



### Покрытие каскадного связывания и профиль загрузки `RAW` (для SLA)

Оцениваем вклад резервного ключа `hero_id_fallback` и измеряем реальную глубину истории и ежедневный прирост данных в `pg__matches`.

**Выводы:**
* **Эффективность каскадного `JOIN`:**
  * Связано по `account_id` (`is_matched_match = true`): 198 508 941 событие (80,48%, 8,27 млрд урона).
  * Спасено по `hero_id_fallback` (`is_matched_match = true`): 46 811 081 событие (18,98%, 1,94 млрд урона).
  * Внутри согласованных матчей (`is_matched_match = true`) несопоставленных событий нет (100% покрытие). Оставшиеся 0,54% событий относятся к матчам вне `pg__matches`.
* **Глубина истории и режим пополнения (321 997 матчей в `pg__matches`):**
  * Игровое время (`start_time`): 2026-10-14 12:44 - 2026-10-15 08:06 (~19,5 часов игр).
  * Окно загрузки (`sys_insert_stamp`): 2026-10-18 - 2026-11-14 (28 дней ретроспективного допарсинга, от 1 830 до 17 941 матча в день). Это обосновывает необходимость инкремента по `sys_insert_stamp`, а не по дате матча `start_time`.

In [3]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window

with SparkSessionContext("lakehouse.stage_case13") as spark:
    st_tde = spark.table("lakehouse.stage.case13_tower_damage")
    w_all = Window.partitionBy()
    
    print("=== 1. Покрытие каскадного JOIN и несопоставленные записи ===")
    st_tde.groupBy("player_match_method", "is_matched_match").agg(
        F.count("*").alias("events"),
        F.sum(F.col("damage").cast("long")).alias("dmg")
    ).withColumn("events_pct", F.round(F.col("events") * 100.0 / F.sum("events").over(w_all), 2)) \
     .orderBy(F.desc("events")).show(truncate=False)
    
    print("=== 2. Глубина истории и ежедневный прирост (pg__matches, ~320k строк) ===")
    spark.table("lakehouse.raw.pg__matches").select(
        F.from_unixtime("start_time").cast("timestamp").alias("match_dt"),
        "sys_insert_stamp"
    ).agg(
        F.min("match_dt").alias("min_match_dt"),
        F.max("match_dt").alias("max_match_dt"),
        F.min("sys_insert_stamp").alias("min_ingest_ts"),
        F.max("sys_insert_stamp").alias("max_ingest_ts"),
        F.count("*").alias("matches_cnt")
    ).show(truncate=False)
    
    spark.table("lakehouse.raw.pg__matches") \
        .groupBy(F.to_date("sys_insert_stamp").alias("ingest_date")) \
        .agg(F.count("*").alias("matches_cnt")) \
        .orderBy(F.desc("ingest_date")).show(15, truncate=False)

=== 1. Покрытие каскадного JOIN и несопоставленные записи ===


26/11/14 08:54:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:54:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:54:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:54:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:54:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 08:54:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 0

+-------------------+----------------+---------+----------+----------+
|player_match_method|is_matched_match|events   |dmg       |events_pct|
+-------------------+----------------+---------+----------+----------+
|account_id         |true            |198508941|8267081175|80.48     |
|hero_id_fallback   |true            |46811081 |1944540008|18.98     |
|account_id         |false           |514528   |22171642  |0.21      |
|unmatched          |false           |501334   |21803148  |0.2       |
|hero_id_fallback   |false           |326830   |13952780  |0.13      |
+-------------------+----------------+---------+----------+----------+

=== 2. Глубина истории и ежедневный прирост (pg__matches, ~320k строк) ===


+-------------------+-------------------+--------------------------+--------------------------+-----------+
|min_match_dt       |max_match_dt       |min_ingest_ts             |max_ingest_ts             |matches_cnt|
+-------------------+-------------------+--------------------------+--------------------------+-----------+
|2026-10-14 12:44:45|2026-10-15 08:06:20|2026-10-18 16:33:23.709706|2026-11-14 08:18:49.723377|321997     |
+-------------------+-------------------+--------------------------+--------------------------+-----------+



[Stage 9:=========================================>               (11 + 1) / 15]

+-----------+-----------+
|ingest_date|matches_cnt|
+-----------+-----------+
|2026-11-14 |1830       |
|2026-11-13 |5088       |
|2026-11-12 |3561       |
|2026-11-11 |4969       |
|2026-11-10 |4282       |
|2026-11-09 |8537       |
|2026-11-08 |17941      |
|2026-11-07 |17310      |
|2026-11-06 |16969      |
|2026-11-05 |17076      |
|2026-11-04 |15424      |
|2026-11-03 |15522      |
|2026-11-02 |17036      |
|2026-11-01 |14317      |
|2026-10-31 |15123      |
+-----------+-----------+
only showing top 15 rows
